# Visual Genome predicate triplet statistics

This notebook prints train/test triplet statistics for each scene-graph predicate.

For every predicate, it reports:

- total relation instances in train and test,
- number of unique `(subject, predicate, object)` triplets,
- train/test unique-triplet overlap,
- most frequent train/test triplets.

Use this notebook separately from the HOLa low-rank decomposition notebook so dataset statistics and feature decomposition analysis do not get mixed together.


## 1. Configuration

Edit paths here if your VG files are stored somewhere else.


In [ ]:
import os

REPO_ROOT = "/Users/shangfei/Developer/SDSGG"

VG_ROIDB_FILE = os.path.join(REPO_ROOT, "datasets/vg/VG-SGG-with-attri.h5")
VG_DICT_FILE = os.path.join(REPO_ROOT, "datasets/vg/VG-SGG-dicts-with-attri.json")

# Match the repository's Visual Genome split logic in visual_genome.py.
NUM_VAL_IM = 5000

# Print only these predicates if not None. Example: ["on", "under", "holding"]
SELECTED_PREDICATES = None

# How many top triplets to print for each predicate and split.
TOPK_TRIPLETS_PER_PREDICATE = 10

# If True, print all selected predicates one by one. This can be long for VG's full predicate set.
PRINT_PER_PREDICATE_DETAILS = True

# Optional export paths for compact summaries.
EXPORT_SUMMARY_JSON = os.path.join(REPO_ROOT, "outputs/vg_predicate_triplet_summary.json")
EXPORT_SUMMARY_CSV = os.path.join(REPO_ROOT, "outputs/vg_predicate_triplet_summary.csv")


## 2. Imports and helper functions


In [ ]:
import csv
import json
from typing import Dict, List, Sequence, Tuple

import h5py
import torch

print("VG roidb file:", VG_ROIDB_FILE)
print("VG dict file:", VG_DICT_FILE)
print("NUM_VAL_IM:", NUM_VAL_IM)


In [ ]:
def normalize_key(text: str) -> str:
    return " ".join(str(text).strip().lower().split())


def build_idx_to_name_map(payload: Dict, idx_key: str, to_idx_key: str) -> Dict[int, str]:
    """Build a robust integer-id -> name map.

    Some VG/GQA dict files are 1-based and stored as dicts, while others are lists.
    Using an explicit map is safer than indexing a Python list by raw annotation ids.
    """
    if idx_key in payload:
        values = payload[idx_key]
        if isinstance(values, dict):
            return {int(index): name for index, name in values.items()}
        return {int(index): name for index, name in enumerate(values)}

    if to_idx_key in payload:
        return {int(index): name for name, index in payload[to_idx_key].items()}

    raise ValueError(f"Missing {idx_key} / {to_idx_key} in dict file.")


def load_vg_label_info(dict_file: str):
    """Load robust object/predicate id mappings from the VG dict JSON."""
    with open(dict_file, "r", encoding="utf-8") as handle:
        payload = json.load(handle)

    idx_to_class = build_idx_to_name_map(payload, "idx_to_label", "label_to_idx")
    idx_to_predicate = build_idx_to_name_map(payload, "idx_to_predicate", "predicate_to_idx")

    all_class_names = [name for _, name in sorted(idx_to_class.items(), key=lambda x: x[0])]
    all_predicate_names = [name for _, name in sorted(idx_to_predicate.items(), key=lambda x: x[0])]

    return {
        "idx_to_class": idx_to_class,
        "idx_to_predicate": idx_to_predicate,
        "all_class_names": all_class_names,
        "all_predicate_names": all_predicate_names,
    }


def build_vg_triplet_statistics(roidb_file: str, dict_file: str, num_val_im: int = 5000):
    """Build train/test counts for each (subject, predicate, object) triplet.

    Split logic follows maskrcnn_benchmark/data/datasets/visual_genome.py:
    - train uses split == 0 after skipping the first num_val_im images,
    - test uses split == 2,
    - images without boxes or relations are skipped.
    """
    if not os.path.exists(roidb_file):
        raise FileNotFoundError(
            f"VG roidb file not found: {roidb_file}. Put VG-SGG-with-attri.h5 there or update VG_ROIDB_FILE."
        )
    if not os.path.exists(dict_file):
        raise FileNotFoundError(
            f"VG dict file not found: {dict_file}. Put VG-SGG-dicts-with-attri.json there or update VG_DICT_FILE."
        )

    info = load_vg_label_info(dict_file)
    idx_to_class = info["idx_to_class"]
    idx_to_predicate = info["idx_to_predicate"]
    split_to_triplets = {"train": {}, "test": {}}
    split_to_predicate_counts = {"train": {}, "test": {}}
    split_to_image_counts = {}
    skipped_relations = {"train": 0, "test": 0}

    with h5py.File(roidb_file, "r") as roi_h5:
        data_split = roi_h5["split"][:]
        img_to_first_box = roi_h5["img_to_first_box"][:]
        img_to_last_box = roi_h5["img_to_last_box"][:]
        img_to_first_rel = roi_h5["img_to_first_rel"][:]
        img_to_last_rel = roi_h5["img_to_last_rel"][:]
        all_labels = roi_h5["labels"][:, 0]
        all_relations = roi_h5["relationships"][:]
        all_predicates = roi_h5["predicates"][:, 0]

        def image_indices_for_split(split_name: str):
            split_flag = 2 if split_name == "test" else 0
            split_mask = data_split == split_flag
            split_mask &= img_to_first_box >= 0
            split_mask &= img_to_first_rel >= 0
            image_index = [int(i) for i in (split_mask.nonzero())[0]]
            if split_name == "train":
                image_index = image_index[num_val_im:]
            elif split_name == "val":
                image_index = image_index[:num_val_im]
            return image_index

        for split_name in ["train", "test"]:
            image_index = image_indices_for_split(split_name)
            split_to_image_counts[split_name] = len(image_index)
            predicate_triplets = {}
            predicate_counts = {}

            for img_id in image_index:
                rel_start = int(img_to_first_rel[img_id])
                rel_end = int(img_to_last_rel[img_id])
                box_start = int(img_to_first_box[img_id])
                box_end = int(img_to_last_box[img_id])
                if rel_start < 0:
                    continue

                rel_pairs = all_relations[rel_start: rel_end + 1] - box_start
                rel_preds = all_predicates[rel_start: rel_end + 1]
                gt_classes = all_labels[box_start: box_end + 1]

                for (sub_idx, obj_idx), pred_idx in zip(rel_pairs, rel_preds):
                    pred_idx = int(pred_idx)
                    sub_class_idx = int(gt_classes[sub_idx])
                    obj_class_idx = int(gt_classes[obj_idx])

                    pred_name = idx_to_predicate.get(pred_idx)
                    sub_name = idx_to_class.get(sub_class_idx)
                    obj_name = idx_to_class.get(obj_class_idx)

                    if pred_name is None or sub_name is None or obj_name is None:
                        skipped_relations[split_name] += 1
                        continue

                    triplet = (sub_name, pred_name, obj_name)
                    predicate_triplets.setdefault(pred_name, {})
                    predicate_triplets[pred_name][triplet] = predicate_triplets[pred_name].get(triplet, 0) + 1
                    predicate_counts[pred_name] = predicate_counts.get(pred_name, 0) + 1

            split_to_triplets[split_name] = predicate_triplets
            split_to_predicate_counts[split_name] = predicate_counts

    return {
        "idx_to_class": idx_to_class,
        "idx_to_predicate": idx_to_predicate,
        "all_class_names": info["all_class_names"],
        "all_predicate_names": info["all_predicate_names"],
        "split_to_triplets": split_to_triplets,
        "split_to_predicate_counts": split_to_predicate_counts,
        "split_to_image_counts": split_to_image_counts,
        "skipped_relations": skipped_relations,
    }


def summarize_predicate_triplets(vg_stats, predicate_name: str, topk: int = 10):
    train_triplets = vg_stats["split_to_triplets"]["train"].get(predicate_name, {})
    test_triplets = vg_stats["split_to_triplets"]["test"].get(predicate_name, {})
    train_count = vg_stats["split_to_predicate_counts"]["train"].get(predicate_name, 0)
    test_count = vg_stats["split_to_predicate_counts"]["test"].get(predicate_name, 0)
    train_unique = len(train_triplets)
    test_unique = len(test_triplets)
    overlap = set(train_triplets.keys()) & set(test_triplets.keys())

    return {
        "predicate": predicate_name,
        "train_total": train_count,
        "test_total": test_count,
        "train_unique_triplets": train_unique,
        "test_unique_triplets": test_unique,
        "shared_unique_triplets": len(overlap),
        "train_overlap_ratio": 0.0 if train_unique == 0 else len(overlap) / train_unique,
        "test_overlap_ratio": 0.0 if test_unique == 0 else len(overlap) / test_unique,
        "train_topk": sorted(train_triplets.items(), key=lambda x: x[1], reverse=True)[:topk],
        "test_topk": sorted(test_triplets.items(), key=lambda x: x[1], reverse=True)[:topk],
    }


def triplet_to_string(triplet: Tuple[str, str, str]) -> str:
    return f"{triplet[0]} | {triplet[1]} | {triplet[2]}"


## 3. Build train/test triplet statistics

This may take a little time because it scans all VG relations once.


In [ ]:
print("Step 3: Build Visual Genome train/test triplet statistics.")
print("What this step does:")
print("- Reads subject/object labels and relation annotations from VG.")
print("- Builds counts for each (subject, predicate, object) triplet in train and test.")
print("- Uses the same train/test split convention as this repository's VG dataset loader.")
print("- Uses explicit idx->name dictionaries, so 1-based VG ids are handled correctly.")

vg_triplet_stats = build_vg_triplet_statistics(VG_ROIDB_FILE, VG_DICT_FILE, num_val_im=NUM_VAL_IM)

all_predicate_names = [
    name for name in vg_triplet_stats["all_predicate_names"] if normalize_key(name) != "__background__"
]
if SELECTED_PREDICATES is None:
    predicate_names_to_print = all_predicate_names
else:
    selected = {normalize_key(name) for name in SELECTED_PREDICATES}
    predicate_names_to_print = [name for name in all_predicate_names if normalize_key(name) in selected]

print("Done building VG triplet statistics.")
print("Train images with relations:", vg_triplet_stats["split_to_image_counts"].get("train"))
print("Test images with relations:", vg_triplet_stats["split_to_image_counts"].get("test"))
print("Predicates in dict excluding background:", len(all_predicate_names))
print("Predicates selected for printing:", len(predicate_names_to_print))
print("Predicates with train stats:", len(vg_triplet_stats["split_to_triplets"]["train"]))
print("Predicates with test stats:", len(vg_triplet_stats["split_to_triplets"]["test"]))
print("Skipped relations due to missing id mapping:", vg_triplet_stats["skipped_relations"])

sample_pred_ids = sorted(list(vg_triplet_stats["idx_to_predicate"].keys()))[:10]
print("Sample predicate id->name mapping:", {idx: vg_triplet_stats["idx_to_predicate"][idx] for idx in sample_pred_ids})


## 4. Print per-predicate train/test triplet details

If this prints too much, set `SELECTED_PREDICATES = ["on", "under", ...]` or `PRINT_PER_PREDICATE_DETAILS = False` in the config cell.


In [ ]:
if PRINT_PER_PREDICATE_DETAILS:
    print("Per-predicate train/test triplet statistics")
    print("Meaning of fields:")
    print("- train_total/test_total: total relation instances for this predicate.")
    print("- unique_triplets: distinct subject-predicate-object combinations.")
    print("- test_overlap_ratio: fraction of test unique triplets also seen in train.")

    for predicate_name in predicate_names_to_print:
        summary = summarize_predicate_triplets(
            vg_stats=vg_triplet_stats,
            predicate_name=predicate_name,
            topk=TOPK_TRIPLETS_PER_PREDICATE,
        )
        print("\n" + "=" * 100)
        print("Predicate:", summary["predicate"])
        print({
            "train_total": summary["train_total"],
            "test_total": summary["test_total"],
            "train_unique_triplets": summary["train_unique_triplets"],
            "test_unique_triplets": summary["test_unique_triplets"],
            "shared_unique_triplets": summary["shared_unique_triplets"],
            "train_overlap_ratio": round(summary["train_overlap_ratio"], 4),
            "test_overlap_ratio": round(summary["test_overlap_ratio"], 4),
        })
        print("Top train triplets:")
        for triplet, count in summary["train_topk"]:
            print("  ", {"triplet": triplet, "count": int(count)})
        print("Top test triplets:")
        for triplet, count in summary["test_topk"]:
            print("  ", {"triplet": triplet, "count": int(count)})
else:
    print("PRINT_PER_PREDICATE_DETAILS=False, skip detailed printing.")


## 5. Compact summary rankings

These rankings are useful before reading the long per-predicate printout.


In [ ]:
summary_rows = []
for predicate_name in predicate_names_to_print:
    summary = summarize_predicate_triplets(
        vg_stats=vg_triplet_stats,
        predicate_name=predicate_name,
        topk=TOPK_TRIPLETS_PER_PREDICATE,
    )
    summary_rows.append({
        "predicate": predicate_name,
        "train_total": summary["train_total"],
        "test_total": summary["test_total"],
        "train_unique_triplets": summary["train_unique_triplets"],
        "test_unique_triplets": summary["test_unique_triplets"],
        "shared_unique_triplets": summary["shared_unique_triplets"],
        "train_overlap_ratio": summary["train_overlap_ratio"],
        "test_overlap_ratio": summary["test_overlap_ratio"],
    })

rows_by_train = sorted(summary_rows, key=lambda x: x["train_total"], reverse=True)
rows_by_test_overlap = sorted(summary_rows, key=lambda x: x["test_overlap_ratio"])
rows_by_test_unique = sorted(summary_rows, key=lambda x: x["test_unique_triplets"], reverse=True)

print("Top 15 predicates by train frequency:")
for row in rows_by_train[:15]:
    print(row)

print("\nTop 15 predicates by test unique triplets:")
for row in rows_by_test_unique[:15]:
    print(row)

print("\nTop 15 predicates with lowest test overlap ratio:")
for row in rows_by_test_overlap[:15]:
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})


## 6. Export compact summaries

This exports one row per predicate. Top triplet lists remain in notebook output; the compact files are easier to inspect programmatically.


In [ ]:
os.makedirs(os.path.dirname(EXPORT_SUMMARY_JSON), exist_ok=True)

with open(EXPORT_SUMMARY_JSON, "w", encoding="utf-8") as handle:
    json.dump(summary_rows, handle, ensure_ascii=False, indent=2)

with open(EXPORT_SUMMARY_CSV, "w", encoding="utf-8", newline="") as handle:
    fieldnames = [
        "predicate",
        "train_total",
        "test_total",
        "train_unique_triplets",
        "test_unique_triplets",
        "shared_unique_triplets",
        "train_overlap_ratio",
        "test_overlap_ratio",
    ]
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(summary_rows)

print("Saved JSON summary:", EXPORT_SUMMARY_JSON)
print("Saved CSV summary:", EXPORT_SUMMARY_CSV)
